# Silver Exploration Notebook


## Imports

In [54]:
import os
import findspark

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    lit,
    when,
    to_timestamp,
    from_utc_timestamp,
    dayofweek,
    hour,
    month,
    input_file_name,
    regexp_extract,
    arrays_zip,
    explode,
    size,
    count,
    length,
    min,
    max,
    slice,
    regexp_extract,
    desc,
    sum as spark_sum
)

## Start Spark

In [3]:
findspark.init()

spark = (
    SparkSession.builder
    .appName("silver-exploration")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/26 07:54:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
import os

RAW = os.path.expanduser("~/spark-lab-data/dev/project/raw")

print(RAW)

/home/ubuntu/spark-lab-data/dev/project/raw


## Read datasets

In [5]:
# Bike-sharing data
trips = (
    spark.read
    .option("header", True)
    .csv(f"{RAW}/trips/*/*.csv")
)

# Weather data
weather = (
    spark.read
    .option("multiLine", True)
    .json(f"{RAW}/weather/*/*.json")
)

## Exporation of Bike-sharing dataset

In [13]:
trips.printSchema()
print("Trips:", trips.count())

root
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_description: string (nullable = true)
 |-- start_station_latitude: string (nullable = true)
 |-- start_station_longitude: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_description: string (nullable = true)
 |-- end_station_latitude: string (nullable = true)
 |-- end_station_longitude: string (nullable = true)
 |-- src: string (nullable = false)
 |-- city: string (nullable = false)
 |-- year_month: string (nullable = false)
 |-- started_at_local: timestamp (nullable = true)
 |-- ended_at_local: timestamp (nullable = true)



Trips: 8428152


In [40]:
trips.show(10, truncate=False)

+--------------------------------+--------------------------------+--------+----------------+-------------------------+----------------------------+----------------------+-----------------------+--------------+------------------+---------------------------------+--------------------+---------------------+-------------------------------------------------------------------------+----+----------+-----------------------+-----------------------+
|started_at                      |ended_at                        |duration|start_station_id|start_station_name       |start_station_description   |start_station_latitude|start_station_longitude|end_station_id|end_station_name  |end_station_description          |end_station_latitude|end_station_longitude|src                                                                      |city|year_month|started_at_local       |ended_at_local         |
+--------------------------------+--------------------------------+--------+----------------+-----------------

### Investigating missing values

In [50]:
trips.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in trips.columns
]).show()

+----------+--------+--------+----------------+------------------+-------------------------+----------------------+-----------------------+--------------+----------------+-----------------------+--------------------+---------------------+---+----+----------+----------------+--------------+
|started_at|ended_at|duration|start_station_id|start_station_name|start_station_description|start_station_latitude|start_station_longitude|end_station_id|end_station_name|end_station_description|end_station_latitude|end_station_longitude|src|city|year_month|started_at_local|ended_at_local|
+----------+--------+--------+----------------+------------------+-------------------------+----------------------+-----------------------+--------------+----------------+-----------------------+--------------------+---------------------+---+----+----------+----------------+--------------+
|         0|       0|       0|               0|                 0|                    11000|                     0|            

### Investigating the timestamp format

In [52]:
trips.groupBy(length("started_at").alias("len")).count().orderBy("len").show()
trips.select("started_at", "ended_at", "duration").distinct().limit(10).show(truncate=False)

+---+-------+
|len|  count|
+---+-------+
| 25|   8193|
| 32|8419959|
+---+-------+



26/10/26 09:21:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/26 09:21:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+--------------------------------+--------------------------------+--------+
|started_at                      |ended_at                        |duration|
+--------------------------------+--------------------------------+--------+
|2022-08-01 05:43:19.520000+00:00|2022-08-01 05:49:42.620000+00:00|383     |
|2022-08-01 05:52:50.567000+00:00|2022-08-01 06:01:56.539000+00:00|545     |
|2022-08-01 06:02:50.691000+00:00|2022-08-01 06:25:51.980000+00:00|1381    |
|2022-08-01 06:08:43.742000+00:00|2022-08-01 06:15:05.617000+00:00|381     |
|2022-08-01 06:56:45.359000+00:00|2022-08-01 07:05:30.466000+00:00|525     |
|2022-08-01 10:07:12.028000+00:00|2022-08-01 10:13:42.627000+00:00|390     |
|2022-08-01 11:52:32.729000+00:00|2022-08-01 11:57:46.421000+00:00|313     |
|2022-08-01 12:36:30.210000+00:00|2022-08-01 12:45:30.657000+00:00|540     |
|2022-08-01 13:32:05.073000+00:00|2022-08-01 13:37:42.847000+00:00|337     |
|2022-08-01 13:57:02.466000+00:00|2022-08-01 14:41:58.879000+00:00|2696    |

In [68]:
# Setting the spark session timezone to UTC
spark.conf.set("spark.sql.session.timeZone", "UTC")

In [70]:
trips = (
    trips
    .withColumn("started_at", to_timestamp(col("started_at")))
    .withColumn("ended_at", to_timestamp(col("ended_at")))
)

trips.filter(
    col("started_at").isNull() |
    col("ended_at").isNull()
).count()

0

### Converting Timestamps to Norwegian Local Time

The `started_at` and `ended_at` timestamps are recorded in UTC, as indicated by `+00:00`.

For example:
- `started_at`: 2022-08-01 05:14:13.781000+00:00
- `ended_at`: 2022-08-01 05:26:27.585000+00:00

We convert these timestamps to Norwegian local time (`Europe/Oslo`), accounting for daylight saving time. The original UTC timestamps are preserved for further processing.

In [ ]:
trips = (
    trips
    .withColumn(
        "started_at_local",
        from_utc_timestamp(
            to_timestamp(col("started_at")),
            "Europe/Oslo"
        )
    )
    .withColumn(
        "ended_at_local",
        from_utc_timestamp(
            to_timestamp(col("ended_at")),
            "Europe/Oslo"
        )
    )
)

trips.select("started_at","started_at_local","ended_at","ended_at_local").show(10, truncate=False)

+--------------------------------+-----------------------+--------------------------------+-----------------------+
|started_at                      |started_at_local       |ended_at                        |ended_at_local         |
+--------------------------------+-----------------------+--------------------------------+-----------------------+
|2022-08-01 03:08:45.036000+00:00|2022-08-01 05:08:45.036|2022-08-01 03:11:38.571000+00:00|2022-08-01 05:11:38.571|
|2022-08-01 03:09:05.419000+00:00|2022-08-01 05:09:05.419|2022-08-01 03:18:15.885000+00:00|2022-08-01 05:18:15.885|
|2022-08-01 03:22:25.178000+00:00|2022-08-01 05:22:25.178|2022-08-01 03:30:57.051000+00:00|2022-08-01 05:30:57.051|
|2022-08-01 03:29:20.355000+00:00|2022-08-01 05:29:20.355|2022-08-01 03:38:43.156000+00:00|2022-08-01 05:38:43.156|
|2022-08-01 03:29:25.883000+00:00|2022-08-01 05:29:25.883|2022-08-01 03:37:00.311000+00:00|2022-08-01 05:37:00.311|
|2022-08-01 03:30:15.563000+00:00|2022-08-01 05:30:15.563|2022-08-01 03:

### Investingating the duration parameter 

In [ ]:
t = trips.withColumn("duration_s", col("duration").cast("double"))

t.select("duration_s").summary("min", "25%", "50%", "75%", "max").show()

t.select(spark_sum(when(col("duration_s") <= 0, 1).otherwise(0)).alias("non_positive"),
         spark_sum(when(col("duration_s") < 60, 1).otherwise(0)).alias("under_60s"),
         spark_sum(when(col("duration_s").isNull(), 1).otherwise(0)).alias("null_duration")).show()

+-------+----------+
|summary|duration_s|
+-------+----------+
|    min|       0.0|
|    25%|     341.0|
|    50%|     524.0|
|    75%|     806.0|
|    max|   79790.0|
+-------+----------+



+------------+---------+-------------+
|non_positive|under_60s|null_duration|
+------------+---------+-------------+
|           2|     1273|            0|
+------------+---------+-------------+



#### Durations to investigate ferther
- Is it posseble that a trips last around 22 hours?
- 2 trips has a negative durations

In [ ]:
print("Number of trips longer than 12 hours: ", t.filter(col("duration_s") > 12 * 3600).count())

t.select(
    "started_at",
    "ended_at",
    "duration_s"
).orderBy(desc("duration_s")).show(15, truncate=False)

Number of trips longer than 12 hours:  15


+--------------------------------+--------------------------------+----------+
|started_at                      |ended_at                        |duration_s|
+--------------------------------+--------------------------------+----------+
|2026-06-19 13:38:13.233000+00:00|2026-06-20 11:48:03.076000+00:00|79790.0   |
|2025-10-17 14:35:20.232000+00:00|2025-10-18 11:15:52.091000+00:00|74431.0   |
|2022-04-16 12:47:44.363000+00:00|2022-04-17 07:37:10.178000+00:00|67765.0   |
|2023-04-07 13:40:11.612000+00:00|2023-04-08 08:18:37.147000+00:00|67105.0   |
|2023-05-31 15:31:03.704000+00:00|2023-06-01 08:37:01.096000+00:00|61557.0   |
|2026-05-04 18:28:16.356000+00:00|2026-05-05 11:14:23.840000+00:00|60367.0   |
|2022-11-01 15:19:05.897000+00:00|2022-11-02 07:26:44.435000+00:00|58058.0   |
|2022-11-24 14:20:12.725000+00:00|2022-11-25 05:44:40.784000+00:00|55468.0   |
|2026-08-06 20:38:28.625000+00:00|2026-08-07 11:37:01.191000+00:00|53913.0   |
|2022-05-20 16:49:31.692000+00:00|2022-05-21 06:44:4

+-------+----------+
|summary|duration_s|
+-------+----------+
|    min|       0.0|
|    25%|     341.0|
|    50%|     524.0|
|    75%|     806.0|
|    max|   79790.0|
+-------+----------+



#### Removing trips that has a duration smaller than 0.

In [67]:
trips_silver = t.filter(col("duration_s") > 0)

### Number of trips divided by the different cities

In [17]:
trips = (trips
         .withColumn("src", input_file_name())
         .withColumn("city", regexp_extract("src", r"/trips/([^/]+)/", 1))
         .withColumn("year_month", regexp_extract("src", r"/([0-9]{4}-[0-9]{2})\.csv", 1)))

trips.groupBy("city").count().show()

+---------+-------+
|     city|  count|
+---------+-------+
|     oslo|5718138|
|   bergen|1904236|
|trondheim| 805778|
+---------+-------+



In [22]:
trips.filter(length("started_at") == 25).select("started_at", "ended_at").show(5, truncate=False)

p = (trips
     .withColumn("start_ts", to_timestamp("started_at"))
     .withColumn("end_ts", to_timestamp("ended_at"))
     .withColumn("len_start", length("started_at"))
     .withColumn("len_end", length("ended_at")))

p.groupBy("len_start", "len_end").count().show()
p.filter(col("start_ts").isNull() | col("end_ts").isNull()).count()

from pyspark.sql.functions import regexp_extract, input_file_name, length, count, sum as spark_sum, when

chk = (trips
       .withColumn("src", input_file_name())
       .withColumn("city", regexp_extract("src", r"/trips/([^/]+)/", 1))
       .withColumn("year_month", regexp_extract("src", r"/([0-9]{4}-[0-9]{2})\.csv", 1))
       .groupBy("city", "year_month")
       .agg(count("*").alias("rows"),
            spark_sum(when(length("started_at") == 25, 1).otherwise(0)).alias("short_start")))

chk.withColumn("pct", col("short_start") / col("rows") * 100) \
   .orderBy(col("pct").desc()).show(10)

+-------------------------+--------------------------------+
|started_at               |ended_at                        |
+-------------------------+--------------------------------+
|2022-08-01 13:05:31+00:00|2022-08-01 13:18:14.855000+00:00|
|2022-08-01 14:54:32+00:00|2022-08-01 15:01:39.232000+00:00|
|2022-08-01 16:01:52+00:00|2022-08-01 16:24:06.468000+00:00|
|2022-08-01 16:30:21+00:00|2022-08-01 16:37:31.473000+00:00|
|2022-08-01 17:07:22+00:00|2022-08-01 17:25:54.295000+00:00|
+-------------------------+--------------------------------+
only showing top 5 rows



+---------+-------+-------+
|len_start|len_end|  count|
+---------+-------+-------+
|       32|     32|8411899|
|       32|     25|   8060|
|       25|     32|   8188|
|       25|     25|      5|
+---------+-------+-------+



+---------+----------+-----+-----------+-------------------+
|     city|year_month| rows|short_start|                pct|
+---------+----------+-----+-----------+-------------------+
|trondheim|   2024-06|21822|         35|0.16038859866190083|
|   bergen|   2025-07|28903|         43|  0.148773483721413|
|     oslo|   2025-12|24317|         34|0.13981987909692808|
|   bergen|   2023-10|42989|         60| 0.1395705878247924|
|trondheim|   2024-07|17209|         24| 0.1394619094659771|
|trondheim|   2023-05|16792|         23|0.13696998570747976|
|trondheim|   2026-03| 2952|          4|0.13550135501355012|
|   bergen|   2026-02|11837|         16|0.13516938413449353|
|   bergen|   2023-03|30451|         41|0.13464254047486124|
|   bergen|   2026-06|25541|         34|0.13311929838299205|
+---------+----------+-----+-----------+-------------------+
only showing top 10 rows



In [23]:
t = trips.withColumn("duration_s", col("duration").cast("double"))
print(t.approxQuantile("duration_s", [0.5, 0.99, 0.999, 0.9999], 0.001))
print(t.filter(col("duration_s") > 6*3600).count())
print(t.filter(col("duration_s") > 12*3600).count())
print(t.approxQuantile("duration_s", [0.999, 0.9999], 0.0001))

[524.0, 3706.0, 79790.0, 79790.0]


315


15


[9142.0, 79790.0]


In [24]:
total = trips.count()
distinct = trips.dropDuplicates().count()
print(total, distinct, total - distinct)

26/10/26 08:20:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/26 08:20:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/26 08:20:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/26 08:20:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/26 08:20:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/26 08:20:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/26 08:20:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/26 08:20:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/26 08:20:39 WARN RowBasedKeyValueBatch: Calling spill() on

8428152 8428127 25


## Exporation of Weather dataset

In [10]:
weather.printSchema()
print("Weather files:", weather.count())

root
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- hourly: struct (nullable = true)
 |    |-- apparent_temperature: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- cloud_cover: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- precipitation: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- rain: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- relative_humidity_2m: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- snowfall: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- temperature_2m: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- time: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- weather_code: array (nullable = true)
 |    |    |-- el

In [8]:
from pyspark.sql.functions import arrays_zip, explode

w = (weather
     .withColumn("src", input_file_name())
     .select("src", explode(arrays_zip(
         col("hourly.time"),
         col("hourly.temperature_2m"),
         col("hourly.rain"),
         col("hourly.wind_speed_10m"),
        col("hourly.apparent_temperature"),
        col("hourly.precipitation")
     )).alias("h")))

w.select("src", "h.*").show(3, truncate = False)

+-----------------------------------------------------------------------------------------------------------------------+----------------+--------------+----+--------------+--------------------+-------------+
|src                                                                                                                    |time            |temperature_2m|rain|wind_speed_10m|apparent_temperature|precipitation|
+-----------------------------------------------------------------------------------------------------------------------+----------------+--------------+----+--------------+--------------------+-------------+
|file:///home/ubuntu/spark-lab-data/dev/project/raw/weather/trondheim/2022-01-01_2026-10-18_20261019T112729Z-52a0eb.json|2022-01-01T00:00|0.5           |0.0 |7.7           |-3.0                |0.0          |
|file:///home/ubuntu/spark-lab-data/dev/project/raw/weather/trondheim/2022-01-01_2026-10-18_20261019T112729Z-52a0eb.json|2022-01-01T01:00|0.2           |0.0 |7.6   

In [29]:
weather.select(
    "timezone",
    slice("hourly.time", 1, 5).alias("first_timestamps")
).show(10, truncate=False)

+--------+------------------------------------------------------------------------------------------+
|timezone|first_timestamps                                                                          |
+--------+------------------------------------------------------------------------------------------+
|GMT     |[2022-01-01T00:00, 2022-01-01T01:00, 2022-01-01T02:00, 2022-01-01T03:00, 2022-01-01T04:00]|
|GMT     |[2022-01-01T00:00, 2022-01-01T01:00, 2022-01-01T02:00, 2022-01-01T03:00, 2022-01-01T04:00]|
|GMT     |[2022-01-01T00:00, 2022-01-01T01:00, 2022-01-01T02:00, 2022-01-01T03:00, 2022-01-01T04:00]|
|GMT     |[2026-10-09T00:00, 2026-10-09T01:00, 2026-10-09T02:00, 2026-10-09T03:00, 2026-10-09T04:00]|
|GMT     |[2026-10-09T00:00, 2026-10-09T01:00, 2026-10-09T02:00, 2026-10-09T03:00, 2026-10-09T04:00]|
|GMT     |[2026-10-06T00:00, 2026-10-06T01:00, 2026-10-06T02:00, 2026-10-06T03:00, 2026-10-06T04:00]|
|GMT     |[2026-10-09T00:00, 2026-10-09T01:00, 2026-10-09T02:00, 2026-10-09T03:00,

In [31]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

weather_silver = (
    weather
    .withColumn(
        "hourly_data",
        explode(arrays_zip(
            "hourly.time",
            "hourly.temperature_2m",
            "hourly.precipitation",
            "hourly.wind_speed_10m"
        ))
    )
    .select(
        col("latitude"),
        col("longitude"),
        col("hourly_data.time").alias("time"),
        col("hourly_data.temperature_2m").alias("temperature_2m"),
        col("hourly_data.precipitation").alias("precipitation"),
        col("hourly_data.wind_speed_10m").alias("wind_speed_10m")
    )
    .withColumn("time_utc", to_timestamp("time"))
    .withColumn(
        "time_local",
        from_utc_timestamp("time_utc", "Europe/Oslo")
    )
    .drop("time")
)

weather_silver.show(10, truncate=False)
weather_silver.printSchema()

+---------+---------+--------------+-------------+--------------+-------------------+-------------------+
|latitude |longitude|temperature_2m|precipitation|wind_speed_10m|time_utc           |time_local         |
+---------+---------+--------------+-------------+--------------+-------------------+-------------------+
|63.444637|10.366492|0.5           |0.0          |7.7           |2022-01-01 00:00:00|2022-01-01 01:00:00|
|63.444637|10.366492|0.2           |0.0          |7.6           |2022-01-01 01:00:00|2022-01-01 02:00:00|
|63.444637|10.366492|-0.2          |0.0          |5.9           |2022-01-01 02:00:00|2022-01-01 03:00:00|
|63.444637|10.366492|-0.7          |0.0          |5.5           |2022-01-01 03:00:00|2022-01-01 04:00:00|
|63.444637|10.366492|-3.6          |0.0          |6.9           |2022-01-01 04:00:00|2022-01-01 05:00:00|
|63.444637|10.366492|-1.2          |0.0          |5.4           |2022-01-01 05:00:00|2022-01-01 06:00:00|
|63.444637|10.366492|-0.7          |0.0       

##### Check if any weather values are missing

In [19]:
from pyspark.sql.functions import sum as spark_sum, when

weather_cols = ["temperature_2m", "rain", "wind_speed_10m",
                "apparent_temperature", "precipitation"]

wx.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in weather_cols
]).show()

+--------------+----+--------------+--------------------+-------------+
|temperature_2m|rain|wind_speed_10m|apparent_temperature|precipitation|
+--------------+----+--------------+--------------------+-------------+
|             0|   0|             0|                   0|            0|
+--------------+----+--------------+--------------------+-------------+

